<a href="https://colab.research.google.com/github/mohameddhameem/ASR-Model-Evaluation/blob/main/WhisperV3Large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade transformers datasets[audio] accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 31.1 MB/s eta 0:00:00


In [11]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Using device: {device}")

def get_speech_pipe(model_id):
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
    ).to(device)
    processor = AutoProcessor.from_pretrained(model_id)
    return pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        return_timestamps=True
    )

Using device: cuda:0


In [26]:
# Load Dataset
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

dataset_new = load_dataset("jlvdoorn/atco2-asr-atcosim", "default", split="validation")
newsample = dataset_new[0]["audio"]


data/train-00000-of-00005-c6681348ac8543(…):   0%|          | 0.00/406M [00:00<?, ?B/s]

data/train-00001-of-00005-464e7b29cac82c(…):   0%|          | 0.00/407M [00:00<?, ?B/s]

data/train-00002-of-00005-008f8516235177(…):   0%|          | 0.00/401M [00:00<?, ?B/s]

data/train-00003-of-00005-13846616069619(…):   0%|          | 0.00/387M [00:00<?, ?B/s]

data/train-00004-of-00005-0565e63298f50d(…):   0%|          | 0.00/419M [00:00<?, ?B/s]

data/validation-00000-of-00002-7a5ea3756(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

data/validation-00001-of-00002-56cef5651(…):   0%|          | 0.00/245M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8092 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2026 [00:00<?, ? examples/s]

In [27]:
# Inspect the feature metadata to see the expected structure
print(f"Audio Feature Info: {dataset_new.features['audio']}")

# Force decoding by accessing the 'array' key via the dataset's __getitem__ logic
# We use 'dataset[0]' which should trigger the decoder for all features in the row
sample_row = dataset_new[0]
audio_dict = sample_row['audio']

# If it still returns a decoder, we can manually call the decode method if available
# or check the ndim of the array if it's already a numpy array.
if hasattr(audio_dict, 'keys'):
    print(f"Array shape: {audio_dict['array'].shape}")
    channels = "Mono" if audio_dict['array'].ndim == 1 else "Stereo"
    print(f"Detected: {channels}")
    print(f"Sampling rate: {audio_dict['sampling_rate']} Hz")
else:
    # Fallback for newer versions of datasets with torchcodec
    # We extract the array via indexing which usually forces the conversion
    print("Manual extraction from decoder object...")
    audio_array = audio_dict['array']
    sampling_rate = audio_dict['sampling_rate']
    channels = "Mono" if audio_array.ndim == 1 else "Stereo"
    print(f"Array shape: {audio_array.shape}")
    print(f"Detected: {channels}")
    print(f"Sampling rate: {sampling_rate} Hz")

Audio Feature Info: Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)
Manual extraction from decoder object...
Array shape: (195584,)
Detected: Mono
Sampling rate: 16000 Hz


In [29]:
import torchaudio

# Load the local wav file
file_path = "/content/test.mp3"
waveform, sample_rate = torchaudio.load(file_path)

# Check dimensions
# Torchaudio returns [channels, time]
num_channels = waveform.shape[0]
channel_type = "Mono" if num_channels == 1 else "Stereo"

print(f"File: {file_path}")
print(f"Waveform shape: {waveform.shape}")
print(f"Detected: {channel_type} ({num_channels} channel(s))")
print(f"Sampling Rate: {sample_rate} Hz")

File: /content/test.mp3
Waveform shape: torch.Size([2, 58116096])
Detected: Stereo (2 channel(s))
Sampling Rate: 44100 Hz


In [13]:
# Inference with Whisper Large V3
pipe_v3 = get_speech_pipe("openai/whisper-large-v3")
result_v3 = pipe_v3(sample)
display(result_v3)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

{'text': " Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel. Nor is Mr. Quilter's manner less interesting than his matter. He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the mind. He has grave doubts whether Sir Frederick Leighton's work is really Greek after all, One can discover in it but little of rocky Ithaca. Linnell's pictures are a sort of Upgards and Adam paintings, and Mason's exquisite idylls are as national as a jingo poem. Mr. Burkett Foster's landscapes smile at one much in the same way that Mr. Carker used to flash his teeth, and Mr. John Collier gives his sitter a cheerful slap on the back before he says, like a shampooer in a Turkish bath. Next man!",
 'chunks': [{'timestamp': (0.0, 6.6),
   'text': ' Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.'},
  {'timestamp': (6.6, 1

In [14]:
# Inference with Whisper Large V3 Turbo
pipe_turbo = get_speech_pipe("openai/whisper-large-v3-turbo")
result_turbo = pipe_turbo(sample)
display(result_turbo)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

{'text': " Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel. Nor is Mr. Quilter's manner less interesting than his matter. He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the mind. He has grave doubts whether Sir Frederick Layton's work is really Greek after all, and can discover in it but little of rocky Ithaca. Linnell's pictures are a sort of Up Guards and Adam paintings, and Mason's exquisite idles are as national as a jingo poem. Mr. Birkett Foster's landscapes smile at one much in the same way that Mr. Carker used to flash his teeth, and Mr. John Collier gives his sitter a cheerful slap on the back before he says like a shampooer in a Turkish bath next man",
 'chunks': [{'timestamp': (0.0, 5.3),
   'text': ' Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.'},
  {'timestamp': (6.36, 10.0

In [30]:
# Inference on the stereo test.mp3 file using both pipelines
test_file = "/content/test.mp3"

print(f"Running inference on: {test_file}")

print("\n--- Whisper Large V3 Results ---")
# Note: Whisper models expect 16kHz mono. The pipeline handles resampling/mixing automatically.
result_v3_stereo = pipe_v3(test_file)
print(result_v3_stereo['text'][:500], "...") # Displaying first 500 chars

print("\n--- Whisper Large V3 Turbo Results ---")
result_turbo_stereo = pipe_turbo(test_file)
print(result_turbo_stereo['text'][:500], "...")

Running inference on: /content/test.mp3

--- Whisper Large V3 Results ---


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


 English Leap Podcast. From Speak English with Klaas. Hey, English learners. Welcome back to the English Leap Podcast. Your English podcast for daily life English, real conversations, and easy English listening practice. Yeah, this is the place where you learn real English from real talk, not from boring grammar books. I'm Anna. And I'm Jake. So, Jake, how are you today? Did you have a busy day? Yeah, a little busy. I had work in the morning, and then I went to the gym. But I feel good now. How  ...

--- Whisper Large V3 Turbo Results ---
 English Leap Podcast. From Speak English with Class. Hey English learners, welcome back to the English Leap Podcast, your English podcast for daily life English, real conversations, and easy English listening practice. Yeah, this is the place where you learn real English from real talk, not from boring grammar books. I'm Anna. And I'm Jake. So Jake, how are you today? Did you have a busy day? Yeah, a little busy. I had work in the morning, and then I

In [31]:
result_v3_stereo

{'text': " English Leap Podcast. From Speak English with Klaas. Hey, English learners. Welcome back to the English Leap Podcast. Your English podcast for daily life English, real conversations, and easy English listening practice. Yeah, this is the place where you learn real English from real talk, not from boring grammar books. I'm Anna. And I'm Jake. So, Jake, how are you today? Did you have a busy day? Yeah, a little busy. I had work in the morning, and then I went to the gym. But I feel good now. How about you, Anna? I'm good, but a bit tired. I slipped late last night because I was watching a movie. Ah, so you were maybe wasting a little time? Maybe just a little, but it was a nice movie, and today I'm happy because we can talk with our listeners again. Same here. It always feels nice to sit down, talk with you, and help people with their English. Yes. It feels like we are all in one big room together. Okay. Small talk finished. Now let's get into our topic. Yes. Before we start w

In [32]:
result_turbo_stereo

{'text': " English Leap Podcast. From Speak English with Class. Hey English learners, welcome back to the English Leap Podcast, your English podcast for daily life English, real conversations, and easy English listening practice. Yeah, this is the place where you learn real English from real talk, not from boring grammar books. I'm Anna. And I'm Jake. So Jake, how are you today? Did you have a busy day? Yeah, a little busy. I had work in the morning, and then I went to the gym. But I feel good now. How about you, Anna? I'm good, but a bit tired. I slipped late last night because I was watching a movie. Ah, so you were maybe wasting a little time? Maybe just a little, but it was a nice movie. And today I'm happy because we can talk with our listeners again. Same here. It always feels nice to sit down, talk with you, and help people with their English. Yes, it feels like we are all in one big room together. Okay, small talk finished. Now let's get into our topic. Yes, before we start was

In [33]:
!pip install -q nemo_toolkit[asr]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.1/443.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.0 MB/s eta 0:0

### Speaker Diarization with NVIDIA NeMo
NeMo uses a modular approach. We will use a pre-trained Clustering Diarizer model. Unlike Whisper, NeMo diarization typically requires a `.yaml` configuration file to define the pipeline.

In [50]:
import os
import json
import torch
from pydub import AudioSegment
from nemo.collections.asr.models import ClusteringDiarizer
from omegaconf import OmegaConf

# 1. Convert MP3 to WAV (16kHz Mono)
def prepare_audio(input_path):
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    out_path = "input_audio_16k.wav"
    audio.export(out_path, format="wav")
    return out_path

wav_path = prepare_audio("/content/test.mp3")

# 2. Create Manifest
def create_manifest(audio_path):
    manifest_path = 'manifest.json'
    with open(manifest_path, 'w') as f:
        metadata = {
            "audio_filepath": os.path.abspath(audio_path),
            "offset": 0,
            "duration": None,
            "label": "infer",
            "text": "-",
            "num_speakers": None,
            "rttm_filepath": None
        }
        f.write(json.dumps(metadata) + "\n")
    return manifest_path

manifest = create_manifest(wav_path)

# 3. Build Configuration Programmatically
config = OmegaConf.create({
    'diarizer': {
        'manifest_filepath': manifest,
        'out_dir': "diarization_output",
        'oracle_vad': False,
        'vad': {
            'model_path': 'vad_multilingual_marblenet',
            'parameters': {
                'window_length_in_sec': 0.15,
                'shift_length_in_sec': 0.01,
                'smoothing': 'median',
                'overlap': 0.5,
                'onset': 0.1,
                'offset': 0.1,
                'pad_onset': 0.1,
                'Clarify': 0.1,
                'min_duration_on': 0.2,
                'min_duration_off': 0.2,
                'filter_speech_first': True
            }
        },
        'speaker_embeddings': {
            'model_path': "titanet_large",
            'parameters': {
                'window_length_in_sec': 1.5,
                'shift_length_in_sec': 0.75,
                'multiscale_weights': [1, 1, 1],
                'save_embeddings': False
            }
        },
        'clustering': {
            'parameters': {
                'oracle_num_speakers': False,
                'max_num_speakers': 20,
                'enhanced_mag_threshold': 1.0,
                'sparse_search_volume': 30
            }
        }
    },
    'device': "cuda" if torch.cuda.is_available() else "cpu",
    'sample_rate': 16000,
    'num_workers': 4
})

# 4. Run Diarization
diarizer = ClusteringDiarizer(cfg=config)
diarizer.diarize()

[NeMo I 2026-05-03 00:30:59 clustering_diarizer:117] Loading pretrained vad_multilingual_marblenet model from NGC
[NeMo I 2026-05-03 00:30:59 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.3/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2026-05-03 00:30:59 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.3/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo
[NeMo I 2026-05-03 00:30:59 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-05-03 00:30:59 classification_models:641] Please use the EncDecSpeakerLabelModel instead of this model. EncDecClassificationModel model is kept for backward compatibility with older models.
[NeMo W 2026-05-03 00:30:59 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/ami_train_0.63.json,/manifests/freesound_background_train.json,/manifests/freesound_laughter_train.json,/manifests/fisher_2004_background.json,/manifests/fisher_2004_speech_sampled.json,/manifests/google_train_manifest.json,/manifests/icsi_all_0.63.json,/manifests/musan_freesound_train.json,/manifests/musan_music_train.json,/manifests/musan_soundbible_train.json,/manifests/mandarin_train_sample.json,/manifests/german_train_sample.json,/manifests/spanish_train_sample.json,/manifests/french_train_sample.json,/manifests/russian_tr

[NeMo I 2026-05-03 00:30:59 save_restore_connector:285] Model EncDecClassificationModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2026-05-03 00:30:59 clustering_diarizer:150] Loading pretrained titanet_large model from NGC
[NeMo I 2026-05-03 00:30:59 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.
[NeMo I 2026-05-03 00:30:59 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2026-05-03 00:30:59 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-05-03 00:31:02 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-05-03 00:31:02 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method 

[NeMo I 2026-05-03 00:31:02 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.


[NeMo W 2026-05-03 00:31:02 clustering_diarizer:398] Deleting previous clustering diarizer outputs.


[NeMo I 2026-05-03 00:31:02 speaker_utils:92] Number of files to diarize: 1
[NeMo I 2026-05-03 00:31:02 clustering_diarizer:303] Split long audio file to avoid CUDA memory issue


splitting manifest: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]


[NeMo I 2026-05-03 00:31:04 vad_utils:146] The prepared manifest file exists. Overwriting!
[NeMo I 2026-05-03 00:31:04 classification_models:594] Perform streaming frame-level VAD
[NeMo I 2026-05-03 00:31:04 collections:750] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2026-05-03 00:31:04 collections:751] Dataset successfully loaded with 27 items and total duration provided from manifest is  0.37 hours.
[NeMo I 2026-05-03 00:31:04 collections:757] # 27 files loaded accounting to # 1 labels


AttributeError: 'ClusteringDiarizer' object has no attribute 'verbose'

In [44]:
import os
import glob
import pandas as pd

# List all contents of the output directory to debug
print("Contents of diarization_output:")
!ls -R diarization_output

def parse_rttm(rttm_path):
    """Parses NeMo RTTM output into a list of segments with start, end, and speaker."""
    segments = []
    with open(rttm_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            start = float(parts[3])
            duration = float(parts[4])
            speaker = parts[7]
            segments.append({"start": start, "end": start + duration, "speaker": speaker})
    return segments

def align_transcription(whisper_chunks, rttm_segments):
    aligned = []
    for chunk in whisper_chunks:
        c_start, c_end = chunk['timestamp']
        center = (c_start + c_end) / 2
        current_speaker = "Unknown"
        for seg in rttm_segments:
            if seg['start'] <= center <= seg['end']:
                current_speaker = seg['speaker']
                break
        aligned.append({"speaker": current_speaker, "text": chunk['text'], "start": c_start})
    return aligned

# Dynamically find the RTTM file
rttm_files = glob.glob("diarization_output/pred_rttms/*.rttm")

if rttm_files:
    rttm_file = rttm_files[0]
    print(f"\nProcessing: {rttm_file}")
    segments = parse_rttm(rttm_file)
    final_transcript = align_transcription(result_v3_stereo['chunks'], segments)
    for line in final_transcript:
        print(f"[{line['speaker']}]: {line['text']}")
else:
    print("\nNo RTTM file found yet. If cell e7e551a3 is still showing a spinning icon, please wait for it to finish. If it finished, check for errors in the logs.")

Contents of diarization_output:
diarization_output:
pred_rttms  speaker_outputs

diarization_output/pred_rttms:

diarization_output/speaker_outputs:

No RTTM file found yet. If cell e7e551a3 is still showing a spinning icon, please wait for it to finish. If it finished, check for errors in the logs.
